In [2]:
!pip install catboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 8.6 MB/s eta 0:00:00


#Importing Libraries

In [26]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GroupShuffleSplit

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    matthews_corrcoef,
    balanced_accuracy_score,
    brier_score_loss
)

from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
import random

In [7]:
random.seed(42)

# Load and Split dataset into train and test set

In [8]:
ckd_df = pd.read_csv("/content/CKD_preprocessed.csv")

ckd_X = ckd_df.drop("d_status", axis=1)
ckd_y = ckd_df["d_status"]

In [9]:
ckd_X_train, ckd_X_test, ckd_y_train, ckd_y_test = train_test_split(
    ckd_X,
    ckd_y,
    test_size=0.20,
    random_state=42,
    stratify=ckd_y
)

# Baseline Models

In [13]:
models = {
    "Majority Dummy": DummyClassifier(strategy="most_frequent"),

    "Stratified Dummy": DummyClassifier(strategy="stratified",
                                        random_state=42),

    "Logistic Regression":
        LogisticRegression(max_iter=1000,
                           random_state=42),

    "Decision Tree":
        DecisionTreeClassifier(random_state=42),

    "Random Forest":
        RandomForestClassifier(random_state=42),

    "XGBoost":
        XGBClassifier(
            eval_metric="logloss",
            random_state=42
        ),

    "LightGBM":
        LGBMClassifier(verbose=-1, random_state=42),

    "CatBoost":
        CatBoostClassifier(
            verbose=0,
            random_state=42
        ),

    "SVM":
        SVC(probability=True, kernel='rbf',
            random_state=42),

    "MLP":
        MLPClassifier(
            max_iter=500,
            random_state=42
        )
}

# Evaluation Function

In [14]:
def evaluate_model(model, X_train, X_test, y_train, y_test):

    model.fit(X_train, y_train)

    predictions = model.predict(X_test)

    probabilities = model.predict_proba(X_test)[:,1]

    tn, fp, fn, tp = confusion_matrix(
        y_test,
        predictions
    ).ravel()

    specificity = tn / (tn + fp)

    results = {

        "Accuracy":
            accuracy_score(y_test, predictions),

        "ROC AUC":
            roc_auc_score(y_test, probabilities),

        "PR AUC":
            average_precision_score(y_test, probabilities),

        "F1":
            f1_score(y_test, predictions),

        "MCC":
            matthews_corrcoef(y_test, predictions),

        "Sensitivity":
            recall_score(y_test, predictions),

        "Specificity":
            specificity,

        "Balanced Accuracy":
            balanced_accuracy_score(
                y_test,
                predictions
            ),

        "Brier Score":
            brier_score_loss(
                y_test,
                probabilities
            )
    }

    return results

# Train Every Model for CKD

In [15]:
ckd_results = []

for name, model in models.items():

    print(name)

    scores = evaluate_model(
        model,
        ckd_X_train,
        ckd_X_test,
        ckd_y_train,
        ckd_y_test
    )

    scores["Model"] = name

    ckd_results.append(scores)

Majority Dummy
Stratified Dummy
Logistic Regression
Decision Tree
Random Forest
XGBoost
LightGBM
CatBoost
SVM
MLP


# CKD Results DF

In [16]:
ckd_results_df = pd.DataFrame(ckd_results)

ckd_results_df = ckd_results_df[
    [
        "Model",
        "Accuracy",
        "ROC AUC",
        "PR AUC",
        "F1",
        "MCC",
        "Sensitivity",
        "Specificity",
        "Balanced Accuracy",
        "Brier Score"
    ]
]

ckd_results_df.sort_values(
    by="ROC AUC",
    ascending=False
)

,Model,Accuracy,ROC AUC,PR AUC,F1,MCC,Sensitivity,Specificity,Balanced Accuracy,Brier Score
7,CatBoost,0.934211,0.989583,0.994244,0.949495,0.858641,0.979167,0.857143,0.918155,0.042829
2,Logistic Regression,0.934211,0.989583,0.994267,0.948454,0.857919,0.958333,0.892857,0.925595,0.044095
6,LightGBM,0.960526,0.988839,0.993617,0.969697,0.916698,1.000000,0.892857,0.946429,0.036906
4,Random Forest,0.947368,0.988839,0.993434,0.959184,0.886658,0.979167,0.892857,0.936012,0.051782
5,XGBoost,0.934211,0.988095,0.993304,0.950495,0.862517,1.000000,0.821429,0.910714,0.045096
9,MLP,0.947368,0.985119,0.992095,0.958333,0.886905,0.958333,0.928571,0.943452,0.048315
3,Decision Tree,0.907895,0.889881,0.890695,0.929293,0.800583,0.958333,0.821429,0.889881,0.092105
8,SVM,0.750000,0.887649,0.929389,0.834783,0.479872,1.000000,0.321429,0.660714,0.147886
1,Stratified Dummy,0.592105,0.550595,0.656433,0.686869,0.103892,0.708333,0.392857,0.550595,0.407895
0,Majority Dummy,0.631579,0.500000,0.631579,0.774194,0.000000,1.000000,0.000000,0.500000,0.368421


# Save CKD Results

In [17]:
ckd_results_df.to_csv(
    "CKD_Dataset_Baseline_Results.csv",
    index=False
)

# Load and Split dataset into train and test set

In [50]:
import pandas as pd

d_ckd_df = pd.read_csv("/content/DiabeticCKD_preprocessed.csv")

groups_df = pd.read_csv("/content/Patient_Groups.csv")

In [51]:
d_ckd_df.columns

Index(['Gender', 'Job', 'Family_Background_of_Diabetes', 'Diabetic_Year',
       'Age', 'Follow_suggested_Diet', 'Take_Medicine_for_Diabetes',
       'Take_Insulin', 'Hypertension', 'Heart_Disease', 'Sleep',
       'Water_Consumption', 'Smoke', 'Zarda_Betel_Leaf', 'Walk_Regularly',
       'Urination_Properly', 'Urinary_Infection', 'Pain_killer',
       'Calorie_Intake', 'CKD', 'Current_BMI'],
      dtype='object')

In [52]:
groups = groups_df["Patient_ID"]

In [53]:
d_ckd_X = d_ckd_df.drop(columns=["CKD"])
d_ckd_y = d_ckd_df["CKD"]

In [60]:
from sklearn.model_selection import GroupShuffleSplit

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(
        d_ckd_X,
        d_ckd_y,
        groups=groups
    )
)

d_ckd_X_train = d_ckd_X.iloc[train_idx]
d_ckd_X_test = d_ckd_X.iloc[test_idx]

d_ckd_y_train = d_ckd_y.iloc[train_idx]
d_ckd_y_test = d_ckd_y.iloc[test_idx]

# Verify the split

In [55]:
train_groups = set(groups.iloc[train_idx])
test_groups = set(groups.iloc[test_idx])

print(len(train_groups & test_groups))

0


# Train Every Model for Diabetic CKD

In [56]:
d_ckd_results = []

for name, model in models.items():

    print(name)

    scores = evaluate_model(
        model,
        d_ckd_X_train,
        d_ckd_X_test,
        d_ckd_y_train,
        d_ckd_y_test
    )

    scores["Model"] = name

    d_ckd_results.append(scores)

Majority Dummy
Stratified Dummy
Logistic Regression


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Decision Tree
Random Forest
XGBoost
LightGBM
CatBoost
SVM
MLP


# Diabetic CKD Results DF

In [57]:
d_ckd_results_df = pd.DataFrame(d_ckd_results)

d_ckd_results_df = d_ckd_results_df[
    [
        "Model",
        "Accuracy",
        "ROC AUC",
        "PR AUC",
        "F1",
        "MCC",
        "Sensitivity",
        "Specificity",
        "Balanced Accuracy",
        "Brier Score"
    ]
]

d_ckd_results_df.sort_values(
    by="ROC AUC",
    ascending=False
)

,Model,Accuracy,ROC AUC,PR AUC,F1,MCC,Sensitivity,Specificity,Balanced Accuracy,Brier Score
6,LightGBM,0.90875,0.762664,0.227582,0.098765,0.108612,0.057971,0.989056,0.523514,0.077308
7,CatBoost,0.91375,0.748528,0.198899,0.080000,0.128118,0.043478,0.995896,0.519687,0.077721
5,XGBoost,0.89250,0.742025,0.195768,0.104167,0.065864,0.072464,0.969904,0.521184,0.085268
4,Random Forest,0.91250,0.731973,0.162805,0.000000,-0.010869,0.000000,0.998632,0.499316,0.078651
2,Logistic Regression,0.91375,0.703364,0.153917,0.000000,0.000000,0.000000,1.000000,0.500000,0.076467
9,MLP,0.91375,0.622950,0.142060,0.000000,0.000000,0.000000,1.000000,0.500000,0.090542
1,Stratified Dummy,0.83625,0.523216,0.091424,0.132450,0.042977,0.144928,0.901505,0.523216,0.163750
3,Decision Tree,0.83000,0.513234,0.088811,0.116883,0.024112,0.130435,0.896033,0.513234,0.170000
0,Majority Dummy,0.91375,0.500000,0.086250,0.000000,0.000000,0.000000,1.000000,0.500000,0.086250
8,SVM,0.91375,0.472710,0.078383,0.000000,0.000000,0.000000,1.000000,0.500000,0.079200


# Save Diabetic CKD Results

In [58]:
d_ckd_results_df.to_csv(
    "Diabetic_CKD_Dataset_Baseline_Results.csv",
    index=False
)

# Additional Baseline

In [61]:
baseline_features = [
    "Age",
    "Diabetic_Year"
]

d_ckd_X_baseline = d_ckd_df[baseline_features]

d_ckd_X_train_baseline = d_ckd_X_baseline.iloc[train_idx]
d_ckd_X_test_baseline = d_ckd_X_baseline.iloc[test_idx]

d_ckd_y_train_baseline = d_ckd_y.iloc[train_idx]
d_ckd_y_test_baseline = d_ckd_y.iloc[test_idx]

In [62]:
baseline_model = LogisticRegression(max_iter=1000, random_state=42)

baseline_model.fit(d_ckd_X_train_baseline, d_ckd_y_train_baseline)

LogisticRegression(max_iter=1000, random_state=42)

In [63]:
add_base_score = evaluate_model(
    baseline_model,
    d_ckd_X_train_baseline,
    d_ckd_X_test_baseline,
    d_ckd_y_train_baseline,
    d_ckd_y_test_baseline
    )

In [64]:
add_base_score

{'Accuracy': 0.91375,
 'ROC AUC': np.float64(0.6086758262455639),
 'PR AUC': np.float64(0.10797011208537255),
 'F1': 0.0,
 'MCC': 0.0,
 'Sensitivity': 0.0,
 'Specificity': np.float64(1.0),
 'Balanced Accuracy': np.float64(0.5),
 'Brier Score': np.float64(0.07893997072867796)}